# 1. Import Libraries

In [12]:
import math
from pulp import LpProblem, LpVariable, LpStatus, lpSum, LpMaximize, LpInteger, LpContinuous, value
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import sympy as sp
import pulp
import random
import warnings
from IPython.display import display
import pandas as pd
import numpy as np
import pulp as pl
from itertools import product

warnings.filterwarnings('ignore')


# 2. Read Data Files 

In [13]:
bom = pd.read_csv('data/bill_of_materials.csv')
cakes = pd.read_csv('data/cakes.csv')
channels = pd.read_csv('data/channels.csv')
ingredients = pd.read_csv('data/ingredients.csv')
demand_params = pd.read_csv('data/instructor_demand_competition.csv')
wages_energy = pd.read_csv('data/wages_energy.csv')
price_table = pd.read_csv('data/price_table_template.csv')

# 3. Extract Wages and Cost Parameters

In [14]:
ingredient_cost = ingredients.set_index('ingredient')['unit_cost_usd']
usage = bom.set_index('cake_id')[ingredients['ingredient'].tolist()].fillna(0)
cake_info = cakes.set_index('cake_id')
channel_info = channels.set_index('channel')
w_params = wages_energy.set_index('parameter')['value']
prep_wage_per_minute = float(w_params['prep_wage_usd_per_hour']) / 60
oven_wage_per_minute = float(w_params['oven_wage_usd_per_hour']) / 60
pack_wage_per_minute = float(w_params['pack_wage_usd_per_hour']) / 60
oven_rental_per_minute = float(w_params['oven_rental_usd_per_hour']) / 60
oven_cost_per_minute = float(w_params['oven_cost_usd_per_hour']) / 60
budget = float(w_params['budget_usd'])

# 4. ILP Solver with P set to price table

In [15]:
# ==== Parameter Preparation (added to fix undefined references) ====
cakes_list = cake_info.index.tolist()
channels_list = channel_info.index.tolist()
service_cap = channel_info['service_cap_per_week']  # channel capacity
transport_cost = channel_info['transport_cost_per_unit_usd']
batch_size = cake_info['batch_size_units']
prep_time_per_unit = cake_info['prep_min_per_unit']
pack_time_per_unit = cake_info['pack_min_per_unit']
pack_cost_per_unit = cake_info['packaging_cost_per_unit_usd']
min_prod = cake_info['minimum_units_if_made']
oven_time_per_batch = cake_info['oven_min_per_batch']


# Ingredient unit cost per cake unit (sum of usage * ingredient cost)
cost_ing = (usage * ingredient_cost).sum(axis=1).to_dict()

# Wages & costs (rename to variables expected later in code)
wage_prep = prep_wage_per_minute * 60  # keep hourly equivalent naming
wage_pack = pack_wage_per_minute * 60
wage_decor = wage_prep  # no decor wage provided; assume same as prep
oven_rental_per_min = oven_rental_per_minute
electricity_per_min = oven_cost_per_minute


# Price pivot (fill NaN with 0 so model runs; user can update price_table.csv)
price_pivot = price_table.pivot(index='cake', columns='channel', values='price').reindex(index=cakes_list, columns=channels_list)
price_pivot = price_pivot.fillna(19.0)


# Demand parameters pivot
alpha_pivot = demand_params.pivot(index='ID', columns='channel', values='alpha').reindex(index=cakes_list, columns=channels_list)
beta_pivot = demand_params.pivot(index='ID', columns='channel', values='beta').reindex(index=cakes_list, columns=channels_list)


# Linear demand D = alpha - beta * price (clipped at >=0)
dem = (alpha_pivot - beta_pivot * price_pivot).clip(lower=0)


# ==== Original Model Code (corrected loops & references) ====
m = pl.LpProblem("Sweet_Market_Simple_ILP", pl.LpMaximize)
# Decision variables
y = pl.LpVariable.dicts("y", (cakes_list, channels_list), lowBound=0, cat=pl.LpInteger)
s = pl.LpVariable.dicts("s", (cakes_list, channels_list), lowBound=0, cat=pl.LpInteger)
b = pl.LpVariable.dicts("b", cakes_list, lowBound=0, cat=pl.LpInteger)


# Demand and sales linkage
for i in cakes_list:
    for j in channels_list:
        Dij = float(dem.loc[i, j])
        m += s[i][j] <= Dij
        m += s[i][j] <= y[i][j]


# Channel service capacity
for j in channels_list:
    m += pl.lpSum(s[i][j] for i in cakes_list) <= float(service_cap.loc[j])


# Batch size & minimum production (if produced)
# Introduce binary to enforce minimum only if produced
z = pl.LpVariable.dicts('z_make', cakes_list, lowBound=0, upBound=1, cat=pl.LpInteger)
M_big = {i: float(dem.loc[i].sum()) for i in cakes_list}  # upper bound on production if made
for i in cakes_list:
    # batch linking
    m += pl.lpSum(y[i][j] for j in channels_list) == int(batch_size.loc[i]) * b[i]
    # minimum if made
    if int(min_prod.loc[i]) > 0:
        m += pl.lpSum(y[i][j] for j in channels_list) - int(min_prod.loc[i]) * z[i] >= 0
        m += pl.lpSum(y[i][j] for j in channels_list) - M_big[i] * z[i] <= 0
    else:
        # if no minimum required, z just equals whether any produced (optional)
        m += pl.lpSum(y[i][j] for j in channels_list) - M_big[i] * z[i] <= 0


# Revenue
revenue = pl.lpSum(float(price_pivot.loc[i, j]) * s[i][j] for i in cakes_list for j in channels_list)


# Costs (match original variable names)
c_ing = pl.lpSum(float(cost_ing.get(i, 0.0)) * y[i][j] for i in cakes_list for j in channels_list)
c_prep = pl.lpSum(float(prep_time_per_unit.loc[i]) * y[i][j] for i in cakes_list for j in channels_list) * (wage_prep / 60.0)


# Decor time not provided; assume 0 so cost is 0 (preserve structure)
c_decor = 0 * pl.lpSum(y[i][j] for i in cakes_list for j in channels_list) * (wage_decor / 60.0)
c_pack_labor = pl.lpSum(float(pack_time_per_unit.loc[i]) * y[i][j] for i in cakes_list for j in channels_list) * (wage_pack / 60.0)
c_pack_mat = pl.lpSum(float(pack_cost_per_unit.loc[i]) * y[i][j] for i in cakes_list for j in channels_list)
total_oven_minutes = pl.lpSum(float(oven_time_per_batch.loc[i]) * b[i] for i in cakes_list)
c_oven = total_oven_minutes * (oven_rental_per_min + electricity_per_min)
c_transport = pl.lpSum(float(transport_cost.loc[j]) * s[i][j] for i in cakes_list for j in channels_list)
total_cost = c_ing + c_prep + c_decor + c_pack_labor + c_pack_mat + c_oven + c_transport


# Objective
m += revenue - total_cost

# Budget constraint
m += total_cost <= budget

# Solve
_ = m.solve(pl.PULP_CBC_CMD(msg=False))
status = pl.LpStatus[m.status]
profit = pl.value(m.objective)
print("Status:", status)
print("Profit: ${:,.2f}".format(profit))


# Pack results
rows = []
for i in cakes_list:
    for j in channels_list:
        rows.append({
            'cake': i,
            'channel': j,
            'price': float(price_pivot.loc[i, j]),
            'demand_cap': float(dem.loc[i, j]),
            'produced_y': int(pl.value(y[i][j])),
            'sold_s': int(pl.value(s[i][j]))
        })
df_plan = pd.DataFrame(rows)
df_plan.to_csv('results/plan.csv', index=False)
df_batches = pd.DataFrame({
    'cake': cakes_list,
    'batches': [int(pl.value(b[i])) for i in cakes_list]
})
df_batches.to_csv('results/batches.csv', index=False)
def v(x): return float(pl.value(x))
breakdown = {
    'Revenue': v(revenue),
    'Ingredients': v(c_ing),
    'Prep labor': v(c_prep),
    'Decor labor': v(c_decor),
    'Packaging labor': v(c_pack_labor),
    'Packaging material': v(c_pack_mat),
    'Oven (rental+energy)': v(c_oven),
    'Transportation': v(c_transport),
    'Total cost': v(total_cost),
    'Profit': v(revenue - total_cost)
}
pd.DataFrame(list(breakdown.items()), columns=['Item', 'Value ($)']).to_csv('results/costs.csv', index=False)
print('Saved: results/plan.csv, results/batches.csv, results/costs.csv')

Status: Optimal
Profit: $4,230.89
Saved: results/plan.csv, results/batches.csv, results/costs.csv


In [16]:
# 5. Compute P = [[p0, p1] for i in I, j in J] using existing parameters
I = len(cakes_list)
J = len(channels_list)

# Build ordered arrays consistent with cakes_list and channels_list
cost_ingredients = [float(cost_ing[i]) for i in cakes_list]
labor_cost = [
    float(prep_time_per_unit.loc[i]) * (wage_prep / 60.0) +
    float(pack_time_per_unit.loc[i]) * (wage_pack / 60.0) for i in cakes_list]
utilities_cost = [
    (float(oven_time_per_batch.loc[i]) / float(batch_size.loc[i])) * (oven_rental_per_min + electricity_per_min) for i in cakes_list]
transportation_cost = [float(transport_cost.loc[j]) for j in channels_list]


# alpha and beta as nested lists aligned with [i][j]
alpha = [[float(alpha_pivot.loc[i, j]) for j in channels_list] for i in cakes_list]
beta = [[float(beta_pivot.loc[i, j]) for j in channels_list] for i in cakes_list]

# Initialize P
P = [[None for _ in range(J)] for _ in range(I)]
eps = 1e-12  # tolerance for effectively zero
for i_idx, i in enumerate(cakes_list):
    for j_idx, j in enumerate(channels_list):
        # cost-based intercept p0 (using provided structure)
        p0 = cost_ingredients[i_idx] - labor_cost[i_idx] - utilities_cost[i_idx] - transportation_cost[j_idx]
        # protect against zero / near-zero alpha
        a = alpha[i_idx][j_idx]
        if a <= eps:
            raise ValueError(f"alpha[{i_idx}][{j_idx}] is zero or negative.")
        else:
            b = beta[i_idx][j_idx]
            if abs(b) <= eps:
                raise ValueError(f"beta[{i_idx}][{j_idx}] is zero or near zero.")
            p1 = a / b
        P[i_idx][j_idx] = [p0, p1]

        
# Print all lists
for i_idx in range(I):
    for j_idx in range(J):
        print(f"P[{i_idx}][{j_idx}] = {P[i_idx][j_idx]}")

P[0][0] = [1.6692222222222226, 31.511453179402604]
P[0][1] = [1.4692222222222227, 16.184206095302237]
P[0][2] = [1.2692222222222225, 25.151562910428158]
P[1][0] = [1.4831666666666667, 33.053642485460664]
P[1][1] = [1.2831666666666668, 14.127162993238855]
P[1][2] = [1.0831666666666666, 24.54833663572844]
P[2][0] = [2.700666666666666, 31.805493523894597]
P[2][1] = [2.500666666666666, 16.01897803139296]
P[2][2] = [2.300666666666666, 21.3613496008435]
P[3][0] = [2.095833333333333, 34.24521354933726]
P[3][1] = [1.8958333333333335, 15.412570088806804]
P[3][2] = [1.6958333333333333, 22.753309324508468]
P[4][0] = [6.121, 29.87179487179487]
P[4][1] = [5.921, 13.68832902158313]
P[4][2] = [5.721000000000001, 20.34972292388729]
P[5][0] = [3.976, 33.571166095051446]
P[5][1] = [3.7760000000000002, 14.286200877080939]
P[5][2] = [3.576, 22.676841320760833]
P[6][0] = [0.6356666666666666, 35.21969080553295]
P[6][1] = [0.43566666666666654, 13.214095472239295]
P[6][2] = [0.23566666666666658, 22.7345174579

In [17]:
# 7. Coarse-to-fine search (deterministic) over a single price variable
import math
from typing import Callable, List, Tuple

def coarse_to_fine_maximize(f: Callable[[float], float], lo: float, hi: float, steps: List[float]) -> Tuple[float, float]:
    """
    Deterministically maximize f(x) over [lo, hi] using staged grids.
    For each step in 'steps' (e.g., [1.0, 0.01]), scan lo:step:hi, take top-2 points,
    then zoom-in to span between those for the next (finer) step.
    Returns (x_best, f_best).
    """
    if hi < lo:
        lo, hi = hi, lo
    x_best, f_best = None, -float('inf')
    cur_lo, cur_hi = float(lo), float(hi)
    for k, step in enumerate(steps):
        if step <= 0:
            raise ValueError('Step sizes must be positive.')
        xs = list(np.arange(cur_lo, cur_hi + 0.5 * step, step))
        # Ensure hi is included due to floating point issues
        if xs and xs[-1] < cur_hi - 1e-12:
            xs.append(cur_hi)
        evals = []
        for x in xs:
            val = f(float(x))
            evals.append((val, float(x)))
            if val > f_best:
                f_best, x_best = val, float(x)
        # pick top-2 distinct x by value, then x as tiebreaker for determinism
        evals.sort(key=lambda t: (t[0], -t[1]), reverse=True)
        top = []
        seen = set()
        for val, x in evals:
            if x in seen:
                continue
            top.append((val, x))
            seen.add(x)
            if len(top) == 2:
                break
        # tighten bounds for next stage
        if len(top) == 1:
            cur_lo = cur_hi = top[0][1]
        elif len(top) >= 2:
            x_low = min(top[0][1], top[1][1])
            x_high = max(top[0][1], top[1][1])
            cur_lo, cur_hi = x_low, x_high
    return x_best, f_best

def _solve_profit_for_price_df(price_df: pd.DataFrame) -> Tuple[str, float]:
    dem_cur = (alpha_pivot - beta_pivot * price_df).clip(lower=0)
    m_loc = pl.LpProblem('ILP_c2f', pl.LpMaximize)
    y_loc = pl.LpVariable.dicts('y', (cakes_list, channels_list), 0, None, pl.LpInteger)
    s_loc = pl.LpVariable.dicts('s', (cakes_list, channels_list), 0, None, pl.LpInteger)
    b_loc = pl.LpVariable.dicts('b', cakes_list, 0, None, pl.LpInteger)
    z_loc = pl.LpVariable.dicts('z_make', cakes_list, 0, 1, pl.LpInteger)
    for i in cakes_list:
        for j in channels_list:
            Dij = float(dem_cur.loc[i, j])
            m_loc += s_loc[i][j] <= Dij
            m_loc += s_loc[i][j] <= y_loc[i][j]
    for j in channels_list:
        m_loc += pl.lpSum(s_loc[i][j] for i in cakes_list) <= float(service_cap.loc[j])
    M_big_cur = {i: float(dem_cur.loc[i].sum()) for i in cakes_list}
    for i in cakes_list:
        m_loc += pl.lpSum(y_loc[i][j] for j in channels_list) == int(batch_size.loc[i]) * b_loc[i]
        if int(min_prod.loc[i]) > 0:
            m_loc += pl.lpSum(y_loc[i][j] for j in channels_list) - int(min_prod.loc[i]) * z_loc[i] >= 0
            m_loc += pl.lpSum(y_loc[i][j] for j in channels_list) - M_big_cur[i] * z_loc[i] <= 0
        else:
            m_loc += pl.lpSum(y_loc[i][j] for j in channels_list) - M_big_cur[i] * z_loc[i] <= 0
    revenue_loc = pl.lpSum(float(price_df.loc[i, j]) * s_loc[i][j] for i in cakes_list for j in channels_list)
    c_ing_loc = pl.lpSum(float(cost_ing.get(i, 0.0)) * y_loc[i][j] for i in cakes_list for j in channels_list)
    c_prep_loc = pl.lpSum(float(prep_time_per_unit.loc[i]) * y_loc[i][j] for i in cakes_list for j in channels_list) * (wage_prep / 60.0)
    c_decor_loc = 0
    c_pack_labor_loc = pl.lpSum(float(pack_time_per_unit.loc[i]) * y_loc[i][j] for i in cakes_list for j in channels_list) * (wage_pack / 60.0)
    c_pack_mat_loc = pl.lpSum(float(pack_cost_per_unit.loc[i]) * y_loc[i][j] for i in cakes_list for j in channels_list)
    total_oven_minutes_loc = pl.lpSum(float(oven_time_per_batch.loc[i]) * b_loc[i] for i in cakes_list)
    c_oven_loc = total_oven_minutes_loc * (oven_rental_per_min + electricity_per_min)
    c_transport_loc = pl.lpSum(float(transport_cost.loc[j]) * s_loc[i][j] for i in cakes_list for j in channels_list)
    total_cost_loc = c_ing_loc + c_prep_loc + c_decor_loc + c_pack_labor_loc + c_pack_mat_loc + c_oven_loc + c_transport_loc
    m_loc += total_cost_loc <= budget
    m_loc += revenue_loc - total_cost_loc
    m_loc.solve(pl.PULP_CBC_CMD(msg=False))
    return pl.LpStatus[m_loc.status], pl.value(revenue_loc - total_cost_loc)

def make_profit_fn_for_pair(i_val, j_val, base_price_df: pd.DataFrame) -> Callable[[float], float]:
    def f(x: float) -> float:
        price_df = base_price_df.copy()
        price_df.loc[i_val, j_val] = float(x)
        status, profit = _solve_profit_for_price_df(price_df)
        return -1e18 if status != 'Optimal' else float(profit)
    return f

# Example usage: optimize price for a single (cake, channel) within its [p0, p1] range
# Choose a pair (change these as needed):
i_example = cakes_list[0]  # first cake
j_example = channels_list[0]  # first channel
i_idx = cakes_list.index(i_example)
j_idx = channels_list.index(j_example)
p_lo, p_hi = sorted(P[i_idx][j_idx])
base_prices = price_pivot.copy()
profit_fn = make_profit_fn_for_pair(i_example, j_example, base_prices)
x_star, val_star = coarse_to_fine_maximize(profit_fn, p_lo, p_hi, steps=[1.0, 0.01])
print(f'Best price for (cake {i_example}, channel {j_example}) = {x_star:.2f} -> profit {val_star:.2f}')
# Save the best price scenario
best_prices_df = base_prices.copy()
best_prices_df.loc[i_example, j_example] = x_star
best_prices_df.to_csv('results/best_prices_c2f_single.csv')
print('Saved results/best_prices_c2f_single.csv')

KeyboardInterrupt: 

In [ ]:
# 8. Coordinate-wise coarse-to-fine pricing optimization (deterministic)
from copy import deepcopy

def optimize_all_prices_coarse_to_fine(steps=[1.0, 0.01], max_passes=5, tol=1e-6):
    price_df = price_pivot.copy()
    status, best_profit = _solve_profit_for_price_df(price_df)
    if status != 'Optimal':
        print('Initial price table not feasible/optimal. Status:', status)
    improved = True
    pass_no = 0
    while improved and pass_no < max_passes:
        improved = False
        pass_no += 1
        print(f'Pass {pass_no}...')
        for i_idx, i in enumerate(cakes_list):
            for j_idx, j in enumerate(channels_list):
                p_lo, p_hi = sorted(P[i_idx][j_idx])
                if not np.isfinite(p_lo) or not np.isfinite(p_hi):
                    continue
                if abs(p_hi - p_lo) < 1e-12:
                    continue
                # Local 1D maximize with other prices fixed
                profit_fn = make_profit_fn_for_pair(i, j, price_df)
                x_star, f_star = coarse_to_fine_maximize(profit_fn, p_lo, p_hi, steps=steps)
                # Update price for this pair
                old_price = float(price_df.loc[i, j])
                price_df.loc[i, j] = float(x_star)
                # Re-evaluate full profit
                status_cur, profit_cur = _solve_profit_for_price_df(price_df)
                if status_cur == 'Optimal' and profit_cur > best_profit + tol:
                    best_profit = profit_cur
                    improved = True
                else:
                    # revert if not improved
                    price_df.loc[i, j] = old_price
        print(f'End of pass {pass_no}: best profit = {best_profit:.2f}')
    return price_df, best_profit

best_prices_all, best_profit_all = optimize_all_prices_coarse_to_fine(steps=[1.0, 0.01], max_passes=4)
print(f'Best overall profit (coordinate-wise c2f): {best_profit_all:.2f}')
best_prices_all.to_csv('results/best_prices_c2f_coord.csv')
# Save demand under best prices
best_demand_all = (alpha_pivot - beta_pivot * best_prices_all).clip(lower=0)
best_demand_all.to_csv('results/best_demand_c2f_coord.csv')
print('Saved results/best_prices_c2f_coord.csv and results/best_demand_c2f_coord.csv')

Pass 1...
End of pass 1: best profit = 5681.48
Pass 2...
End of pass 2: best profit = 5933.55
Pass 3...
End of pass 3: best profit = 5955.39
Pass 4...
End of pass 4: best profit = 5955.39
Best overall profit (coordinate-wise c2f): 5955.39
Saved results/best_prices_c2f_coord.csv and results/best_demand_c2f_coord.csv


In [ ]:
Test Optimization With Prices In Ranges

In [23]:
import numpy as np

def _q(x, decimals=6):
    # quantize to avoid float mismatches between search & lookup
    return float(np.round(x, decimals))

def make_cached_profit_fn_for_pair(i, j, price_df):
    """
    Returns (profit_fn, cache) where:
    - profit_fn(x) -> numeric objective used by the search (profit or -inf if infeasible)
    - cache[xq] -> (status, profit) from the global solver for that x at (i, j)
    """
    cache = {}

    def profit_fn(x: float) -> float:
        xq = _q(x)
        if xq in cache:
            status, profit = cache[xq]
        else:
            test_df = price_df.copy()
            test_df.loc[i, j] = xq
            status, profit = _solve_profit_for_price_df(test_df)
            cache[xq] = (status, profit)

        # For the search objective, ignore non-optimal points
        return profit if status == 'Optimal' else -np.inf

    return profit_fn, cache


In [ ]:
def optimize_all_prices_coarse_to_fine(steps=[1.0, 0.01], max_passes=5, tol=1e-6):
    price_df = price_pivot.copy()
    status, best_profit = _solve_profit_for_price_df(price_df)
    if status != 'Optimal':
        print('Initial price table not feasible/optimal. Status:', status)

    improved = True
    pass_no = 0
    while improved and pass_no < max_passes:
        improved = False
        pass_no += 1
        print(f'Pass {pass_no}...')

        for i_idx, i in enumerate(cakes_list):
            for j_idx, j in enumerate(channels_list):
                p_lo, p_hi = sorted(P[i_idx][j_idx])
                if not np.isfinite(p_lo) or not np.isfinite(p_hi):  continue
                if abs(p_hi - p_lo) < 1e-12:                       continue

                # >>> NEW: cached local objective
                profit_fn, cache = make_cached_profit_fn_for_pair(i, j, price_df)

                # 1D coarse-to-fine search (unchanged)
                x_star, f_star = coarse_to_fine_maximize(profit_fn, p_lo, p_hi, steps=steps)

                # Tentatively update
                old_price = float(price_df.loc[i, j])
                price_df.loc[i, j] = float(x_star)

                # >>> NEW: reuse cached global solve; no second solve
                xq = _q(x_star)
                status_cur, profit_cur = cache.get(xq, (None, None))
                if status_cur is None:
                    # rare fallback if rounding mismatch; keeps behavior correct
                    price_df.loc[i, j] = old_price  # avoid double-counting during fallback
                    test_df = price_df.copy()
                    test_df.loc[i, j] = xq
                    status_cur, profit_cur = _solve_profit_for_price_df(test_df)
                    price_df.loc[i, j] = xq  # re-apply

                if status_cur == 'Optimal' and profit_cur > best_profit + tol:
                    best_profit = profit_cur
                    improved = True
                else:
                    price_df.loc[i, j] = old_price

        print(f'End of pass {pass_no}: best profit = {best_profit:.2f}')

    return price_df, best_profit


best_prices_all, best_profit_all = optimize_all_prices_coarse_to_fine(steps=[1.0, 0.01], max_passes=4)
print(f'Best overall profit (coordinate-wise c2f cache): {best_profit_all:.2f}')
best_prices_all.to_csv('results/best_prices_c2f_coord_cache.csv')
# Save demand under best prices
best_demand_all = (alpha_pivot - beta_pivot * best_prices_all).clip(lower=0)
best_demand_all.to_csv('results/best_demand_c2f_coord_cache.csv')
print('Saved results/best_prices_c2f_coord_cache.csv and results/best_demand_c2f_coord_cache.csv')


Pass 1...
